In [ ]:
# CELL 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted!")


In [ ]:
# CELL 2 — Install dependencies and verify GPU
!pip install -q librosa soundfile scikit-learn tqdm tensorflow-addons

import tensorflow as tf
print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPU Available:", bool(gpus))
if gpus:
    print("✅ GPU is enabled")
else:
    print("⚠️ No GPU detected. Use Runtime -> Change runtime type -> GPU if available.")


In [ ]:
# CELL 3 — Set paths and check dataset
import os, zipfile
ZIP_PATH = "/content/drive/MyDrive/mel_3ch.zip"
BASE = "/content/processed/mel_3ch"  # expected folder after extraction

if os.path.exists(ZIP_PATH) and not os.path.exists(BASE):
    print("📦 Extracting zip ...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content')
    print("✅ Extraction complete.")
else:
    print("Using BASE:", BASE)
    if not os.path.exists(BASE):
        raise FileNotFoundError(f"Dataset folder not found at {BASE}. Update ZIP_PATH or BASE.")


In [ ]:
# CELL 4 — Load dataset and OOD if available
import glob
import numpy as np
from tqdm import tqdm
import os

def load_and_sanitize_dataset(base_dir, ood_dirs=None):
    real_files = sorted(glob.glob(os.path.join(base_dir, "real", "*.npy")))
    fake_files = sorted(glob.glob(os.path.join(base_dir, "fake", "*.npy")))

    X_list, y_list = [], []
    dropped = 0

    def sanitize(arr, fpath):
        if not isinstance(arr, np.ndarray):
            raise ValueError("Loaded object is not numpy array.")
        if arr.shape == (128,128):
            arr = np.stack([arr, arr, arr], axis=-1)
        if arr.shape != (128,128,3):
            raise ValueError(f"Unexpected shape: {arr.shape}")
        if not np.isfinite(arr).all():
            raise ValueError("Array contains NaN or Inf.")
        arr = arr.astype(np.float32)
        for c in range(3):
            ch = arr[:,:,c]
            mn, mx = ch.min(), ch.max()
            if mx - mn > 1e-12:
                arr[:,:,c] = (ch - mn) / (mx - mn)
            else:
                arr[:,:,c] = 0.0
        return arr

    print(f"Found {len(real_files)} real, {len(fake_files)} fake files. Validating...")
    for f in tqdm(real_files, desc="Real"):
        try:
            arr = np.load(f)
            arr = sanitize(arr, f)
            X_list.append(arr); y_list.append(1)
        except Exception as e:
            dropped += 1
            print(f"  Dropped (real): {os.path.basename(f)} -> {e}")

    for f in tqdm(fake_files, desc="Fake"):
        try:
            arr = np.load(f)
            arr = sanitize(arr, f)
            X_list.append(arr); y_list.append(0)
        except Exception as e:
            dropped += 1
            print(f"  Dropped (fake): {os.path.basename(f)} -> {e}")

    # optional: load OOD samples if provided (label -1)
    X_ood, y_ood = [], []
    if ood_dirs:
        for label_name, dpath in ood_dirs.items():
            files = sorted(glob.glob(os.path.join(dpath, "*.npy")))
            for f in files:
                try:
                    arr = np.load(f); arr = sanitize(arr, f)
                    X_ood.append(arr); y_ood.append(-1)  # -1 = OOD
                except Exception as e:
                    print(f"  Dropped (ood {label_name}): {os.path.basename(f)} -> {e}")

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int32)
    print(f"Kept: {len(X)} samples. Dropped: {dropped} samples.")
    if X.shape[0] < 10:
        raise RuntimeError("Too few valid samples after sanitization. Fix dataset or preprocessing.")
    return X, y, np.array(X_ood, dtype=np.float32) if X_ood else None, np.array(y_ood, dtype=np.int32) if y_ood else None

# Example OOD folder structure (optional)
OOD_DIRS = {
    "gemini": "/content/processed/ood/gemini",    # put .npy mels here if you have them
    "phone": "/content/processed/ood/phone",
}

# If you don't have OOD npy files, pass ood_dirs=None
X, y, X_ood, y_ood = load_and_sanitize_dataset(BASE, ood_dirs=None)
print("Raw X shape after load:", X.shape)


In [ ]:
# CELL 5 — Audio augmentation helpers
import numpy as np
import librosa
from scipy.signal import butter, lfilter
import random

def add_gaussian_noise(audio, snr_db=20):
    # add noise to get desired SNR
    rms = np.sqrt(np.mean(audio**2))
    snr = 10**(snr_db/20.0)
    noise_rms = rms / snr
    noise = np.random.normal(0, noise_rms, audio.shape)
    return audio + noise

def random_pitch_shift(audio, sr, max_steps=2):
    steps = np.random.uniform(-max_steps, max_steps)
    return librosa.effects.pitch_shift(audio, sr, steps)

def random_time_stretch(audio, max_rate=0.1):
    ratio = np.random.uniform(1 - max_rate, 1 + max_rate)
    # librosa.effects.time_stretch expects > 1 stretching; handle with resample fallback
    try:
        stretched = librosa.effects.time_stretch(audio, rate=ratio)
        # trim/pad to original length
        if len(stretched) > len(audio):
            stretched = stretched[:len(audio)]
        else:
            stretched = np.pad(stretched, (0, max(0, len(audio)-len(stretched))), 'constant')
        return stretched
    except Exception:
        return audio

def random_clipping(audio, clip_frac=0.03):
    if np.random.rand() < 0.3:
        n = int(len(audio) * clip_frac)
        if n < 1: return audio
        idx = np.random.randint(0, len(audio) - n)
        audio[idx:idx+n] = audio[idx:idx+n] * np.random.uniform(0, 0.2)
    return audio

def bandpass_filter(audio, sr, low=300, high=4000):
    # Butterworth bandpass
    nyq = 0.5 * sr
    lowc = low / nyq
    highc = high / nyq
    b, a = butter(2, [lowc, highc], btype='band')
    return lfilter(b, a, audio)

def random_loudness(audio, db_range=6):
    change_db = np.random.uniform(-db_range, db_range)
    factor = 10 ** (change_db / 20.0)
    return audio * factor

def augment_audio(audio, sr):
    # apply random combination of augmentations
    if np.random.rand() < 0.5:
        audio = add_gaussian_noise(audio, snr_db=np.random.uniform(10,30))
    if np.random.rand() < 0.4:
        audio = random_pitch_shift(audio, sr, max_steps=2)
    if np.random.rand() < 0.3:
        audio = random_time_stretch(audio, max_rate=0.1)
    if np.random.rand() < 0.3:
        audio = bandpass_filter(audio, sr, low=200, high=6000)
    audio = random_clipping(audio)
    audio = random_loudness(audio)
    # final clipping to [-1,1]
    audio = np.clip(audio, -1.0, 1.0)
    return audio


In [ ]:
# CELL 6 — Preprocess audio -> mel (keeps compatibility with your original preprocess)
import numpy as np
from scipy.ndimage import zoom
import soundfile as sf

def spec_augment(mel, num_mask=2, freq_masking_max_percentage=0.15, time_masking_max_percentage=0.15):
    mel_aug = mel.copy()
    num_mels, num_frames = mel_aug.shape
    for _ in range(num_mask):
        # freq mask
        freq_percentage = np.random.uniform(0.0, freq_masking_max_percentage)
        num_freqs = int(freq_percentage * num_mels)
        f0 = np.random.randint(0, max(1, num_mels - num_freqs))
        mel_aug[f0:f0+num_freqs, :] = mel_aug.min()
        # time mask
        time_percentage = np.random.uniform(0.0, time_masking_max_percentage)
        num_times = int(time_percentage * num_frames)
        t0 = np.random.randint(0, max(1, num_frames - num_times))
        mel_aug[:, t0:t0+num_times] = mel_aug.min()
    return mel_aug

def preprocess_audio(file_path=None, audio=None, sr_in=None, sample_rate=16000, duration=2.0,
                     n_fft=2048, hop_length=512, n_mels=128, augment=False):
    """
    Either pass file_path OR (audio, sr_in). Returns (1,128,128,3) array.
    If augment=True, apply audio augmentations before computing mel.
    """
    target_length = int(sample_rate * duration)

    if file_path is not None:
        # robust read
        try:
            audio, sr = sf.read(file_path)
            if audio.ndim > 1:
                audio = np.mean(audio, axis=1)
            if sr != sample_rate:
                audio = librosa.resample(audio.astype(np.float32), orig_sr=sr, target_sr=sample_rate)
                sr = sample_rate
        except Exception as e:
            audio, sr = librosa.load(file_path, sr=sample_rate)
    else:
        if audio is None or sr_in is None:
            raise ValueError("Provide either file_path or (audio, sr_in)")
        audio = audio.astype(np.float32)
        if sr_in != sample_rate:
            audio = librosa.resample(audio, orig_sr=sr_in, target_sr=sample_rate)
        sr = sample_rate

    audio = audio.astype(np.float32)
    # apply augmentations on the raw audio before spectrogram
    if augment:
        audio = augment_audio(audio, sr)

    # trim/pad
    if len(audio) < target_length:
        audio = np.pad(audio, (0, target_length - len(audio)), mode="constant")
    else:
        audio = audio[:target_length]

    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_fft=n_fft, hop_length=hop_length,
                                         n_mels=n_mels, power=2.0)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    # optionally apply specaugment (use small prob for train-time)
    if augment and np.random.rand() < 0.9:
        mel_db = spec_augment(mel_db, num_mask=2)

    # Resize and compute delta channels (same as your original)
    z_row = 128 / mel_db.shape[0]
    z_col = 128 / mel_db.shape[1]
    mel_resized = zoom(mel_db, (z_row, z_col), order=1)

    eps = 1e-8
    c0 = (mel_resized - mel_resized.min()) / (mel_resized.max() - mel_resized.min() + eps)

    delta = librosa.feature.delta(mel_db)
    delta2 = librosa.feature.delta(mel_db, order=2)
    delta_resized = zoom(delta, (128/delta.shape[0], 128/delta.shape[1]), order=1)
    delta2_resized = zoom(delta2, (128/delta2.shape[0], 128/delta2.shape[1]), order=1)

    c1 = (delta_resized - delta_resized.min()) / (delta_resized.max() - delta_resized.min() + eps)
    c2 = (delta2_resized - delta2_resized.min()) / (delta2_resized.max() - delta2_resized.min() + eps)

    mel_3ch = np.stack([c0, c1, c2], axis=-1).astype(np.float32)
    if mel_3ch.shape != (128, 128, 3):
        raise ValueError(f"Preprocessed shape is wrong: {mel_3ch.shape}")
    return np.expand_dims(mel_3ch, axis=0)


In [ ]:
# CELL 7 — Build tf.data pipeline for training with mixup
import tensorflow as tf

def mixup(batch_x, batch_y, alpha=0.2):
    # batch_x: (B,H,W,C) numpy, batch_y: (B,) numpy
    lam = np.random.beta(alpha, alpha)
    B = batch_x.shape[0]
    indices = np.random.permutation(B)
    x2 = batch_x[indices]
    y2 = batch_y[indices]
    mixed_x = lam * batch_x + (1 - lam) * x2
    mixed_y = lam * batch_y + (1 - lam) * y2
    return mixed_x, mixed_y

def np_preprocess_batch(X_batch, y_batch, augment=False):
    # X_batch is array of raw mel arrays? If you have raw audio you'd do preprocess_audio here.
    # If using precomputed mel npy files (128x128x3), we only apply specaugment & normalization
    out_x = []
    out_y = []
    for i in range(len(X_batch)):
        x = X_batch[i]
        if augment and np.random.rand() < 0.8:
            # simple spectrogram-level augmentation: random erase (specaugment already applied sometimes)
            x_copy = x.copy()
            # small frequency/time masks
            num_mels, num_frames, _ = x_copy.shape
            # freq mask
            f0 = np.random.randint(0, max(1, num_mels - 10))
            x_copy[f0:f0+5,:,:] = 0.0
            out_x.append(x_copy)
        else:
            out_x.append(x)
        out_y.append(y_batch[i])
    Xb = np.array(out_x, dtype=np.float32)
    yb = np.array(out_y, dtype=np.float32)  # for mixup allow float labels
    return Xb, yb

def make_dataset(X, y, batch_size=64, augment=False, mixup_prob=0.5):
    # Create dataset from numpy arrays and apply augment + mixup
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    ds = ds.shuffle(buffer_size=len(X), reshuffle_each_iteration=True)
    def _map_np(x, y):
        x_out, y_out = tf.numpy_function(lambda a,b: np_preprocess_batch(np.expand_dims(a,0), np.expand_dims(b,0), augment=augment),
                                        [x, y], [tf.float32, tf.float32])
        # the tf.numpy_function call above returns a single sample shaped (1,H,W,C) so we squeeze it later in batch()
        return tf.squeeze(x_out, axis=0), tf.squeeze(y_out, axis=0)
    ds = ds.map(_map_np, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)

    # apply mixup at batch level via numpy_function
    def _mixup_batch(batch_x, batch_y):
        def apply_mix(x_np, y_np):
            if np.random.rand() < mixup_prob:
                x_m, y_m = mixup(x_np, y_np)
                return x_m.astype(np.float32), y_m.astype(np.float32)
            else:
                return x_np.astype(np.float32), y_np.astype(np.float32)
        x_out, y_out = tf.numpy_function(apply_mix, [batch_x, batch_y], [tf.float32, tf.float32])
        x_out.set_shape([None, 128, 128, 3])
        y_out.set_shape([None])
        return x_out, y_out

    ds = ds.map(_mixup_batch, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


In [ ]:
# CELL 8 — Train/val/test splits and dataset creation
from sklearn.model_selection import train_test_split

# keep your stratified split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=1/3, random_state=42, stratify=y_temp)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

BATCH_SIZE = 64
train_ds = make_dataset(X_train, y_train, batch_size=BATCH_SIZE, augment=True, mixup_prob=0.5)
val_ds = make_dataset(X_val, y_val, batch_size=BATCH_SIZE, augment=False, mixup_prob=0.0)
test_ds = make_dataset(X_test, y_test, batch_size=BATCH_SIZE, augment=False, mixup_prob=0.0)


In [ ]:
# CELL 9 — Build model (choose transfer learning or small CNN)
from tensorflow.keras import layers, models, optimizers, applications

def build_small_cnn(input_shape):
    inp = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, (3,3), padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPool2D((2,2))(x)
    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPool2D((2,2))(x)
    x = layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPool2D((2,2))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    model = models.Model(inp, out)
    model.compile(optimizer=optimizers.Adam(1e-4), loss="binary_crossentropy",
                  metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return model

def build_transfer_model(input_shape, backbone="EfficientNetB0", lr=1e-4):
    inp = layers.Input(shape=input_shape)
    # pre-trained on imagenet; we allow training from scratch if GPUs small — but fine to use weights='imagenet'
    if backbone == "EfficientNetB0":
        base = applications.EfficientNetB0(include_top=False, input_shape=input_shape, weights='imagenet', pooling='avg')
    elif backbone == "MobileNetV2":
        base = applications.MobileNetV2(include_top=False, input_shape=input_shape, weights='imagenet', pooling='avg')
    else:
        raise ValueError("Unknown backbone")
    x = base(inp, training=False)  # ensure batchnorm in inference mode in base initially
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    model = models.Model(inp, out)
    # smaller lr for transfer learning
    model.compile(optimizer=optimizers.Adam(lr), loss="binary_crossentropy",
                  metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return model

# Choose:
USE_TL = True   # set False to use small CNN
input_shape = (128,128,3)
if USE_TL:
    model = build_transfer_model(input_shape, backbone="EfficientNetB0", lr=1e-5)
else:
    model = build_small_cnn(input_shape)
model.summary()


In [ ]:
# CELL 10 — Callbacks and training
import tensorflow as tf
BEST_PATH = "/content/drive/MyDrive/best_model_robust.h5"
FINAL_PATH = "/content/drive/MyDrive/voiceguard_cnn_robust.h5"

checkpoint = tf.keras.callbacks.ModelCheckpoint(BEST_PATH, monitor="val_auc", mode="max", save_best_only=True, verbose=1)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor="val_auc", patience=3, factor=0.5, verbose=1, mode="max")
early = tf.keras.callbacks.EarlyStopping(monitor="val_auc", patience=10, restore_best_weights=True, verbose=1, mode="max")

EPOCHS = 50

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[checkpoint, reduce_lr, early],
    verbose=1
)

# Save final model
model.save(FINAL_PATH)
print(f"✅ Saved final model to {FINAL_PATH}")
print(f"✅ Best checkpoint saved to {BEST_PATH}")


In [ ]:
# CELL 11 — Evaluation on test and OOD
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
import numpy as np
from tensorflow.keras.models import load_model
import pandas as pd

# load best model
if os.path.exists(BEST_PATH):
    model_eval = load_model(BEST_PATH, compile=False)
    print("✅ Loaded best checkpoint:", BEST_PATH)
else:
    print("⚠️ BEST_PATH missing, using in-memory model")
    model_eval = model

# helper to get preds from numpy arrays
def predict_on_numpy(model, X_np, batch_size=64):
    probs = model.predict(X_np, batch_size=batch_size).ravel()
    return probs

# Test set
y_prob_test = predict_on_numpy(model_eval, X_test)
if len(np.unique(y_test)) > 1:
    print("Test ROC AUC:", roc_auc_score(y_test, y_prob_test))
else:
    print("Test set has single class; can't compute ROC AUC")

y_pred_test = (y_prob_test >= 0.5).astype(int)
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_test))
print("Classification Report:")
print(classification_report(y_test, y_pred_test, digits=4))

# Save CSV
out_df = pd.DataFrame({"y_true": y_test, "y_prob": y_prob_test, "y_pred_0.5": y_pred_test})
out_csv = "/content/drive/MyDrive/test_predictions_robust.csv"
out_df.to_csv(out_csv, index=False)
print("✅ Saved test predictions to:", out_csv)

# OOD evaluation (if available)
if 'X_ood' in globals() and X_ood is not None and len(X_ood)>0:
    ood_probs = predict_on_numpy(model_eval, X_ood)
    # show distribution summary
    print("OOD predictions stats: mean_prob=%.4f, std=%.4f, min=%.4f, max=%.4f" % (ood_probs.mean(), ood_probs.std(), ood_probs.min(), ood_probs.max()))
    # Save OOD preds
    pd.DataFrame({"y_true": list(y_ood), "y_prob": ood_probs}).to_csv("/content/drive/MyDrive/ood_predictions.csv", index=False)
    print("✅ Saved OOD predictions")
else:
    print("No OOD dataset found (skip OOD eval)")


In [ ]:
# CELL 12 — Inference helper (uses the same preprocess pipeline)
from tensorflow.keras.models import load_model
MODEL_FOR_INFER = BEST_PATH
_model_infer = load_model(MODEL_FOR_INFER)
print("✅ Inference model loaded:", MODEL_FOR_INFER)

def predict_file_with_confidence(file_path, threshold=0.5, low_conf_margin=0.15):
    # preprocess without augment
    X_in = preprocess_audio(file_path, augment=False)
    prob = float(_model_infer.predict(X_in, batch_size=1).ravel()[0])
    label = "REAL" if prob >= threshold else "FAKE"
    # confidence: if prob within [0.5 - margin, 0.5 + margin] mark uncertain
    if abs(prob - 0.5) < low_conf_margin:
        confidence_tag = "UNCERTAIN"
    else:
        confidence_tag = "CONFIDENT"
    print(f"File: {file_path} | Pred: {label} | Prob: {prob:.4f} | {confidence_tag}")
    return prob, label, confidence_tag

# Example usage:
# predict_file_with_confidence("/content/drive/MyDrive/original/gemini_fake_sample.wav")


In [ ]:
# CELL 13 — Short checklist for the team
print("""
Checklist:
- Add some real OOD npy mels in /content/processed/ood/* to test OOD evaluation.
- If you have raw audio edge samples (Gemini/ElevenLabs/etc), run preprocess_audio(file_path, augment=False)
  and save resulting npy to /content/processed/ood/<label>/*.npy for evaluation.
- For small GPU memory: set USE_TL=False to use the small CNN.
- Tune EPOCHS/BATCH_SIZE depending on overfitting:
    * If val_auc diverges from train_auc -> more augmentation and stronger early stopping.
- For production inference: use predict_file_with_confidence() and treat UNCERTAIN as "human review".
""")
